In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

df = pd.DataFrame({
'StudentID': [1, 2, 3, 4, 5, 6 ],
'City': ['Rajkot','Mumbai','Pune','Rajkot','Delhi','Pune'],
'Education':['Bachelors','Masters','PhD','Bachelors','Masters','PhD'],
'Height_cm': [165, 172, 158, 180, 175, 168],
'Weight_kg': [60, 85, 52, 95, 70, 65 ],
'Score_Midterm': [78, 85, 91, 65, 72, 88 ],
'Score_Final': [82, 80, 95, 70, 68, 90 ],
'EnrollDate': pd.to_datetime(['2023-06-01','2022-09-15',
'2023-01-10','2021-03-22',
'2022-07-04','2023-08-19']),
'Pass': [1, 0, 1, 0, 1, 1], # target
})

#creation
df['BMI'] = (df['Weight_kg']/(df['Height_cm']/100)**2).round(2)
df['Score_Improvement'] = df['Score_Final'] - df['Score_Midterm']
df['Score_Total'] = df['Score_Midterm'] + df['Score_Final']

ref = pd.Timestamp('2024-01-01')
df['Days_Since_Enroll'] = (ref - df['EnrollDate']).dt.days
df['Enroll_Month'] = df['EnrollDate'].dt.month
df['Month_sin'] = np.sin(2 * np.pi * df['Enroll_Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Enroll_Month'] / 12)

city_dummies = pd.get_dummies(df['City'], prefix='City')
edu_enc = OrdinalEncoder(categories=[['Bachelors','Masters','PhD']])
df['Education_Enc'] = edu_enc.fit_transform(df[['Education']])

drop_cols = ['StudentID','City','Education','EnrollDate','Enroll_Month','Pass']
X = pd.concat([df.drop(columns=drop_cols), city_dummies], axis=1)
y = df['Pass']
print(X)

   Height_cm  Weight_kg  Score_Midterm  Score_Final    BMI  Score_Improvement  \
0        165         60             78           82  22.04                  4   
1        172         85             85           80  28.73                 -5   
2        158         52             91           95  20.83                  4   
3        180         95             65           70  29.32                  5   
4        175         70             72           68  22.86                 -4   
5        168         65             88           90  23.03                  2   

   Score_Total  Days_Since_Enroll     Month_sin     Month_cos  Education_Enc  \
0          160                214  1.224647e-16 -1.000000e+00            0.0   
1          165                473 -1.000000e+00 -1.836970e-16            1.0   
2          186                356  5.000000e-01  8.660254e-01            2.0   
3          135               1015  1.000000e+00  6.123234e-17            0.0   
4          140                54

In [2]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_scaled)
X_vt = pd.DataFrame(vt.transform(X_scaled),
columns=X.columns[vt.get_support()])
print('After VarianceThreshold:', X_vt.shape)

After VarianceThreshold: (6, 15)


In [3]:
kb = SelectKBest(score_func=f_classif, k=6)
kb.fit(X_vt, y)
X_final = pd.DataFrame(kb.transform(X_vt),
columns=X_vt.columns[kb.get_support()])
print('Final feature set:', X_final.columns.tolist())
print(X_final.head())

Final feature set: ['Height_cm', 'Weight_kg', 'BMI', 'Days_Since_Enroll', 'City_Mumbai', 'City_Pune']
   Height_cm  Weight_kg       BMI  Days_Since_Enroll  City_Mumbai  City_Pune
0  -0.658505  -0.761314 -0.735014          -0.846440    -0.447214  -0.707107
1   0.329252   0.943121  1.289932           0.057593     2.236068  -0.707107
2  -1.646262  -1.306734 -1.101260          -0.350792    -0.447214   1.414214
3   1.458117   1.624895  1.468514           1.949429    -0.447214  -0.707107
4   0.752577  -0.079540 -0.486814           0.312397    -0.447214  -0.707107


In [4]:
X_final['Pass'] = y.values
X_final.to_csv('students_feature_engineered_v2.csv', index=False)